# 🏠 Caso California Housing — Selección de variables con VIF y p-values

## 🎯 Objetivo
Construir un modelo de **regresión lineal múltiple** que prediga el **valor mediano de la vivienda** en distritos de California y decidir, con criterios estadísticos, **qué variables conservar**.

## 🧭 Lo que vas a aprender
1. Detectar **multicolinealidad** con el **VIF** (Variance Inflation Factor)
2. Detectar variables **no significativas** con el **p-value**
3. Eliminar variables **una a la vez** y comparar modelos con datos de prueba
4. Interpretar los coeficientes del modelo final

## 📋 El dataset
Cada fila es un **distrito censal** de California (censo de 1990).

| Variable | Descripción |
|---|---|
| `MedInc` | Ingreso mediano del distrito (en decenas de miles de USD: `3.5` = $35,000) |
| `HouseAge` | Edad mediana de las casas (años) |
| `AveRooms` | Promedio de cuartos por vivienda |
| `AveBedrms` | Promedio de recámaras por vivienda |
| `Population` | Población del distrito |
| `AveOccup` | Promedio de personas por vivienda |
| `Latitude` / `Longitude` | Ubicación del distrito |
| **`MedHouseVal`** | 🎯 **Variable objetivo:** valor mediano de la vivienda (en cientos de miles de USD: `2.5` = $250,000) |

## 🛠️ Funciones auxiliares de visualización

Ejecuta la siguiente celda **una sola vez** al inicio. Después sólo tienes que **llamar** a la función que necesites:

| Función | ¿Para qué sirve? |
|---|---|
| `plot_distributions(df, columnas)` | Histograma + boxplot de variables numéricas |
| `plot_frequencies(df, columnas, top_n=None)` | Frecuencia de variables categóricas |
| `plot_correlation_matrix(df, columnas)` | Matriz de correlación |
| `plot_pairplot(df, columnas, color=None)` | Dispersión entre todas las variables numéricas |
| `plot_simple_regression(x, y, results)` | Recta ajustada de un modelo OLS con 1 variable |
| `plot_actual_vs_predicted(y_real, y_pred)` | Valores reales vs predichos |
| `plot_residuals(y_real, y_pred)` | Residuales vs predichos |
| `plot_rfecv(rfecv)` | R² según el número de variables seleccionadas por RFECV |

In [1]:
# Funciones auxiliares de visualización
# Ejecuta esta celda una vez; después sólo llama a las funciones.
import numpy as np
import plotly.express as px
import plotly.graph_objects as go


def plot_distributions(df, columns, nbins=30):
    """Histograma con boxplot marginal para cada variable numérica."""
    for col in columns:
        fig = px.histogram(
            df,
            x=col,
            nbins=nbins,
            marginal='box',
            opacity=0.7,
            title=f'Distribución de {col}'
        )
        fig.update_layout(bargap=0.2)
        fig.show()


def plot_frequencies(df, columns, top_n=None):
    """Gráfica de barras con la frecuencia de cada categoría (top_n limita a las más comunes)."""
    for col in columns:
        freq = df[col].value_counts()
        if top_n:
            freq = freq.head(top_n)
        freq_df = freq.rename_axis(col).reset_index(name='Frecuencia')

        title = f'Frecuencias de {col}'
        if top_n and df[col].nunique() > top_n:
            title += f' (top {top_n})'

        fig = px.bar(freq_df, x=col, y='Frecuencia', title=title)
        fig.update_layout(xaxis={'categoryorder': 'total descending'})
        fig.show()


def plot_correlation_matrix(df, columns):
    """Mapa de calor con la correlación de Pearson entre las variables numéricas."""
    corr = df[columns].corr().round(2)
    fig = px.imshow(
        corr,
        text_auto=True,
        color_continuous_scale='RdBu_r',
        zmin=-1,
        zmax=1,
        title='Matriz de Correlación'
    )
    fig.update_layout(width=750, height=650)
    fig.show()


def plot_pairplot(df, columns, color=None):
    """Matriz de dispersión (pairplot) entre las variables numéricas."""
    fig = px.scatter_matrix(
        df,
        dimensions=columns,
        color=color,
        title='Pairplot de Variables Numéricas',
        labels={col: col.capitalize() for col in columns}
    )
    fig.update_layout(width=1200, height=1200, title_font_size=20)
    fig.update_traces(diagonal_visible=True)
    fig.show()


def plot_simple_regression(x, y, results):
    """Dispersión de una variable vs el objetivo con la recta ajustada por un OLS de 1 variable."""
    b0, b1 = results.params.iloc[0], results.params.iloc[1]
    x_name = getattr(x, 'name', None) or 'x'
    y_name = getattr(y, 'name', None) or 'y'
    x_line = np.linspace(np.min(x), np.max(x), 100)

    fig = px.scatter(
        x=np.asarray(x),
        y=np.asarray(y),
        opacity=0.6,
        labels={'x': x_name, 'y': y_name},
        title=f'{y_name} = {b0:.2f} + ({b1:.4f}) · {x_name}',
        template='plotly_white'
    )
    fig.add_trace(go.Scatter(
        x=x_line,
        y=b0 + b1 * x_line,
        mode='lines',
        name='Recta OLS',
        line=dict(color='red', width=3)
    ))
    fig.show()


def plot_actual_vs_predicted(y_true, y_pred, title='Real vs Predicho'):
    """Valores reales vs predichos; un modelo perfecto cae sobre la diagonal roja."""
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    lo = min(y_true.min(), y_pred.min())
    hi = max(y_true.max(), y_pred.max())

    fig = px.scatter(
        x=y_true,
        y=y_pred,
        opacity=0.5,
        labels={'x': 'Valor real', 'y': 'Valor predicho'},
        title=title,
        template='plotly_white'
    )
    fig.add_shape(
        type='line', x0=lo, y0=lo, x1=hi, y1=hi,
        line=dict(color='red', dash='dash')
    )
    fig.show()


def plot_residuals(y_true, y_pred, title='Residuales vs Predicho'):
    """Residuales vs predichos; buscamos una nube sin patrón alrededor de 0."""
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)

    fig = px.scatter(
        x=y_pred,
        y=y_true - y_pred,
        opacity=0.5,
        labels={'x': 'Valor predicho', 'y': 'Residual (real − predicho)'},
        title=title,
        template='plotly_white'
    )
    fig.add_hline(y=0, line_dash='dash', line_color='red')
    fig.show()


def plot_rfecv(rfecv):
    """R² promedio de validación cruzada según el número de variables que conserva RFECV."""
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=rfecv.cv_results_['n_features'],
        y=rfecv.cv_results_['mean_test_score'],
        mode='lines+markers',
        line=dict(color='steelblue', width=3),
        marker=dict(size=7),
        name='R² promedio (CV)'
    ))
    fig.update_layout(
        title='RFECV — R² según número de variables seleccionadas',
        xaxis_title='Número de variables',
        yaxis_title='R² (validación cruzada)',
        template='plotly_white',
        width=900, height=450
    )
    fig.show()

### 🧰 Funciones del caso

Estas funciones te ahorran código repetido; ejecuta la celda una vez:

| Función | ¿Para qué sirve? |
|---|---|
| `ajustar_ols(X_train, y_train)` | Ajusta un modelo OLS (ya agrega la constante) |
| `predecir(results, X)` | Genera predicciones con el modelo |
| `evaluar_modelo(nombre, results, X_test, y_test)` | R² y RMSE en el conjunto de prueba |
| `calcular_vif(X)` | VIF de cada variable, de mayor a menor |
| `plot_mapa_precios(df)` | Mapa de los distritos coloreado por precio |

In [2]:
# Funciones del caso: ajustar, evaluar y diagnosticar modelos OLS
import numpy as np
import pandas as pd
import plotly.express as px
import statsmodels.api as sm
from sklearn.metrics import r2_score, mean_squared_error
from statsmodels.stats.outliers_influence import variance_inflation_factor


def ajustar_ols(X_train, y_train):
    """Ajusta un modelo OLS (agrega la constante automáticamente)."""
    return sm.OLS(y_train, sm.add_constant(X_train)).fit()


def predecir(results, X):
    """Predice con un modelo OLS ajustado con ajustar_ols()."""
    return results.predict(sm.add_constant(X))


def evaluar_modelo(nombre, results, X_test, y_test):
    """Imprime y regresa las métricas del modelo en el conjunto de prueba."""
    y_pred = predecir(results, X_test)
    metricas = {
        'modelo': nombre,
        'n_variables': X_test.shape[1],
        'R² ajustado (train)': round(results.rsquared_adj, 4),
        'R² (test)': round(r2_score(y_test, y_pred), 4),
        'RMSE (test)': round(np.sqrt(mean_squared_error(y_test, y_pred)), 4),
    }
    print(f"{nombre}: R² test = {metricas['R² (test)']} | RMSE test = {metricas['RMSE (test)']}")
    return metricas


def calcular_vif(X):
    """VIF de cada variable, de mayor a menor (se calcula con constante, igual que el modelo)."""
    X_const = sm.add_constant(X)
    vif = pd.DataFrame({
        'variable': X.columns,
        'VIF': [variance_inflation_factor(X_const.values, i + 1) for i in range(X.shape[1])],
    })
    return vif.sort_values('VIF', ascending=False).round(2).reset_index(drop=True)


def plot_mapa_precios(df):
    """Ubicación de cada distrito coloreada por el valor mediano de la vivienda."""
    fig = px.scatter(
        df,
        x='Longitude',
        y='Latitude',
        color='MedHouseVal',
        color_continuous_scale='Viridis',
        opacity=0.5,
        title='Valor mediano de la vivienda por ubicación',
        labels={'MedHouseVal': 'Valor (x $100k)'},
        template='plotly_white'
    )
    fig.update_yaxes(scaleanchor='x', scaleratio=1)
    fig.update_layout(width=750, height=700)
    fig.show()

## 1️⃣ Cargar los datos

El dataset viene incluido en scikit-learn; la primera vez se descarga automáticamente.

In [3]:
from sklearn.datasets import fetch_california_housing

df = fetch_california_housing(as_frame=True).frame
df.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


## 2️⃣ Conocer los datos

Antes de modelar respondemos tres preguntas: ¿cuántos datos hay?, ¿hay nulos?, ¿qué rangos tienen las variables?

In [4]:
print(df.shape)
df.isnull().sum()

(20640, 9)


MedInc         0
HouseAge       0
AveRooms       0
AveBedrms      0
Population     0
AveOccup       0
Latitude       0
Longitude      0
MedHouseVal    0
dtype: int64

In [5]:
df.describe().round(2)

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
count,20640.00,20640.00,20640.00,20640.00,20640.00,20640.00,20640.00,20640.00,20640.00
mean,3.87,28.64,5.43,1.10,1425.48,3.07,35.63,-119.57,2.07
std,1.90,12.59,2.47,0.47,1132.46,10.39,2.14,2.00,1.15
min,0.50,1.00,0.85,0.33,3.00,0.69,32.54,-124.35,0.15
25%,2.56,18.00,4.44,1.01,787.00,2.43,33.93,-121.80,1.20
50%,3.53,29.00,5.23,1.05,1166.00,2.82,34.26,-118.49,1.80
75%,4.74,37.00,6.05,1.10,1725.00,3.28,37.71,-118.01,2.65
max,15.00,52.00,141.91,34.07,35682.00,1243.33,41.95,-114.31,5.00


✅ **Qué observar:**
- No hay nulos 🎉
- `AveRooms` y `AveOccup` tienen **máximos extremos** (141 cuartos, 1,243 personas por vivienda): son distritos atípicos (hoteles, dormitorios, etc.)
- `MedHouseVal` tiene un **tope en 5.0** ($500,000): el censo registró "500k o más" como 500k. El modelo no podrá predecir bien las casas más caras.

## 3️⃣ Exploración visual

In [6]:
plot_distributions(df, ['MedHouseVal', 'MedInc'])

In [7]:
plot_correlation_matrix(df, df.columns)

In [8]:
plot_mapa_precios(df)

✅ **Qué observar:**
- `MedInc` es la variable más correlacionada con el precio (**0.69**)
- `AveRooms` y `AveBedrms` están muy correlacionadas entre sí (**0.85**): miden casi lo mismo
- `Latitude` y `Longitude` también (**-0.92**), pero por la **forma diagonal** de California, no porque sobren: juntas indican la ubicación
- En el mapa, las casas caras están en la **costa** (Los Ángeles y el Área de la Bahía)

## 4️⃣ Preparar los datos

Separamos la variable objetivo (`y`) de las predictoras (`X`) y dividimos **80% entrenamiento / 20% prueba**.
Todas las variables ya son numéricas: no necesitamos encoding.

In [9]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['MedHouseVal'])
y = df['MedHouseVal']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Train: {X_train.shape} | Test: {X_test.shape}")

Train: (16512, 8) | Test: (4128, 8)


## 5️⃣ Modelo 1: todas las variables

Empezamos con el modelo completo como **punto de comparación**.

In [10]:
modelo_1 = ajustar_ols(X_train, y_train)
print(modelo_1.summary())

                            OLS Regression Results                            
Dep. Variable:            MedHouseVal   R-squared:                       0.613
Model:                            OLS   Adj. R-squared:                  0.612
Method:                 Least Squares   F-statistic:                     3261.
Date:                Mon, 21 Sep 2026   Prob (F-statistic):               0.00
Time:                        14:51:11   Log-Likelihood:                -17998.
No. Observations:               16512   AIC:                         3.601e+04
Df Residuals:                   16503   BIC:                         3.608e+04
Df Model:                           8                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const        -37.0233      0.728    -50.835      0.0

In [11]:
comparacion = []
comparacion.append(evaluar_modelo('1. Todas las variables', modelo_1, X_test, y_test))

1. Todas las variables: R² test = 0.5758 | RMSE test = 0.7456


## 6️⃣ Paso 1 — Multicolinealidad con VIF

El **VIF** mide qué tanto se puede explicar una variable **con las demás**. Si es alto, la variable es redundante y sus coeficientes se vuelven inestables.

| VIF | Interpretación |
|---|---|
| < 5 | ✅ Sin problema |
| 5 – 10 | ⚠️ Multicolinealidad moderada |
| > 10 | ❌ Multicolinealidad severa |

**Regla:** se elimina **una variable a la vez** y se vuelve a calcular el VIF, porque al quitar una, el VIF de las demás cambia.

In [12]:
calcular_vif(X_train)

,variable,VIF
0,Latitude,9.21
1,Longitude,8.88
2,AveRooms,7.92
3,AveBedrms,6.61
4,MedInc,2.54
5,HouseAge,1.24
6,Population,1.13
7,AveOccup,1.01


Hay dos **pares** con VIF alto: `Latitude`/`Longitude` y `AveRooms`/`AveBedrms`.

### 🧪 Experimento A: quitar la variable con mayor VIF (`Latitude`)
La receta automática diría "quita la de mayor VIF". Veamos qué pasa:

In [13]:
cols_a = X_train.columns.drop('Latitude')

modelo_a = ajustar_ols(X_train[cols_a], y_train)
evaluar_modelo('A. Sin Latitude', modelo_a, X_test[cols_a], y_test);

A. Sin Latitude: R² test = 0.5103 | RMSE test = 0.8011


😮 El R² en test **cae de 0.576 a 0.510**. `Latitude` y `Longitude` tienen VIF alto porque están correlacionadas **entre sí**, pero juntas describen la **ubicación**, que es muy importante para el precio (¡recuerda el mapa!).

> 💡 **Lección:** un VIF alto indica **redundancia**, no que la variable sea inútil. Antes de eliminar, pregúntate qué información aporta.

### 🧪 Experimento B: quitar `AveBedrms`
`AveBedrms` sí es redundante: el número de recámaras ya está contenido en el número de cuartos (`AveRooms`).

In [14]:
cols_b = X_train.columns.drop('AveBedrms')

calcular_vif(X_train[cols_b])

,variable,VIF
0,Latitude,8.83
1,Longitude,8.66
2,MedInc,1.31
3,AveRooms,1.28
4,HouseAge,1.24
5,Population,1.13
6,AveOccup,1.01


In [15]:
modelo_2 = ajustar_ols(X_train[cols_b], y_train)
comparacion.append(evaluar_modelo('2. Sin AveBedrms', modelo_2, X_test[cols_b], y_test))

2. Sin AveBedrms: R² test = 0.5823 | RMSE test = 0.7398


✅ El VIF de `AveRooms` bajó de **7.9 a 1.3** y el R² en test **mejoró ligeramente** (0.576 → 0.582), aunque el R² ajustado en train bajó un poco (0.612 → 0.599): lo que importa es cómo predice con datos nuevos.
`Latitude` y `Longitude` quedan con VIF ≈ 8.8: moderado, pero **las conservamos** porque juntas aportan la ubicación.

## 7️⃣ Paso 2 — Significancia con p-values

El **p-value** de cada coeficiente responde: *"si esta variable en realidad no tuviera efecto, ¿qué tan probable sería observar este coeficiente?"*

- **p < 0.05** → la variable es **significativa**: la conservamos
- **p ≥ 0.05** → no hay evidencia de que aporte: **candidata a eliminar**

In [16]:
modelo_2.pvalues.round(4).sort_values(ascending=False)

Population    0.5859
const         0.0000
MedInc        0.0000
HouseAge      0.0000
AveRooms      0.0000
AveOccup      0.0000
Latitude      0.0000
Longitude     0.0000
dtype: float64

`Population` tiene **p ≈ 0.59**: no es significativa. La eliminamos.

In [17]:
cols_final = cols_b.drop('Population')

modelo_final = ajustar_ols(X_train[cols_final], y_train)
comparacion.append(evaluar_modelo('3. Sin AveBedrms ni Population', modelo_final, X_test[cols_final], y_test))

modelo_final.pvalues.round(4)

3. Sin AveBedrms ni Population: R² test = 0.5823 | RMSE test = 0.7398


const        0.0
MedInc       0.0
HouseAge     0.0
AveRooms     0.0
AveOccup     0.0
Latitude     0.0
Longitude    0.0
dtype: float64

## 8️⃣ Comparar modelos

In [18]:
pd.DataFrame(comparacion)

,modelo,n_variables,R² ajustado (train),R² (test),RMSE (test)
0,1. Todas las variables,8,0.6124,0.5758,0.7456
1,2. Sin AveBedrms,7,0.5994,0.5823,0.7398
2,3. Sin AveBedrms ni Population,6,0.5994,0.5823,0.7398


✅ El modelo final usa **6 variables en lugar de 8** y predice **igual de bien** en test (R² ≈ 0.58, RMSE ≈ 0.74).
Con el mismo desempeño, preferimos el modelo **más simple**: es más fácil de explicar y sus coeficientes son más estables.

## 9️⃣ Diagnóstico del modelo final

In [19]:
y_pred_final = predecir(modelo_final, X_test[cols_final])

plot_actual_vs_predicted(y_test, y_pred_final, title='Modelo final — Real vs Predicho (test)')
plot_residuals(y_test, y_pred_final, title='Modelo final — Residuales vs Predicho')

✅ **Qué observar:**
- La **línea horizontal en 5.0** en la gráfica real vs predicho es el **tope** del dataset: el modelo no puede aprender precios mayores a $500k
- Los residuales forman **bandas diagonales** y se abren a la derecha: la relación no es perfectamente lineal. Un modelo lineal explica ~58% de la variación; el resto requiere otras técnicas (transformaciones, árboles, etc.)

## 🔟 Interpretar los coeficientes

In [20]:
modelo_final.params.round(4)

const       -38.9592
MedInc        0.3720
HouseAge      0.0099
AveRooms      0.0192
AveOccup     -0.0033
Latitude     -0.4574
Longitude    -0.4642
dtype: float64

Cada coeficiente es el cambio en `MedHouseVal` cuando la variable sube **1 unidad**, **manteniendo las demás constantes**:

- **`MedInc` = 0.372** → si el ingreso mediano sube **$10,000**, el valor de la vivienda sube **≈ $37,200**
- **`HouseAge` = 0.0099** → cada año adicional de antigüedad suma **≈ $990** (casas antiguas en zonas céntricas)
- **`Latitude` = -0.457** y **`Longitude` = -0.464** → hacia el **norte** y hacia el **este** (tierra adentro) el precio baja
- **`AveOccup` = -0.0033** → más personas por vivienda, ligeramente menor valor

## 📝 Conclusiones
1. Un **VIF alto no obliga a eliminar** la variable: primero entiende por qué es alto
2. Elimina variables **de una en una** y **compara en test**
3. Con desempeño similar, **el modelo más simple gana**

## 🚀 Retos opcionales
- Elimina los distritos atípicos (`AveOccup > 10`) y vuelve a ajustar el modelo final. ¿Mejora?
- Elimina las filas con `MedHouseVal == 5.0` (el tope). ¿Qué pasa con el R²?